In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/11 02:56:44 WARN Utils: Your hostname, duminghui-XPS-13-9370, resolves to a loopback address: 127.0.1.1; using 172.20.10.4 instead (on interface wlp2s0)
26/03/11 02:56:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 02:56:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark.version

'4.1.1'

In [3]:
df = spark.read.parquet('data/raw/yellow/2025/yellow_tripdata_2025-11.parquet')

In [4]:
df

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double]

In [5]:
df = df.repartition(4)

In [6]:
df.write.parquet('data/homework/pq')

In [7]:
df.show(20)

[Stage 4:====================================>                      (5 + 3) / 8]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|              1|         1.24|         1|                 N|         186|    

In [11]:
start_date = "2025-11-15 00:00:00"
end_date = "2025-11-15 23:59:59"

df.filter(
    (df.tpep_pickup_datetime >= start_date) & 
    (df.tpep_pickup_datetime <= end_date)
).count()

162604

In [17]:
from pyspark.sql.functions import col, unix_timestamp
df.withColumn(
    "duration_hours", 
    (unix_timestamp(df.tpep_dropoff_datetime) - unix_timestamp(df.tpep_pickup_datetime)) / 3600
).orderBy(col("duration_hours").desc()).select("duration_hours").first()

Row(duration_hours=90.64666666666666)

In [49]:
df.createOrReplaceTempView('data_trips')

In [40]:
df

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double]

In [50]:
df_yellow_group = spark.sql("""
SELECT 
    PULocationID,
    COUNT(1) AS number_records
FROM
    data_trips
WHERE
    tpep_pickup_datetime >= '2025-11-01 00:00:00' AND
    tpep_pickup_datetime <= '2025-11-30 23:59:59'
GROUP BY
    1
""")

In [28]:
df_zones = spark.read.parquet('zones/')

In [30]:
df_zones

DataFrame[LocationID: string, Borough: string, Zone: string, service_zone: string]

In [51]:
df_result = df_yellow_group.join(df_zones, df_yellow_group.PULocationID == df_zones.LocationID)

In [55]:
df_result \
    .select("Zone", "PULocationID","number_records") \
    .orderBy("number_records") \
    .show()

[Stage 66:===========================================>              (6 + 2) / 8]

+--------------------+------------+--------------+
|                Zone|PULocationID|number_records|
+--------------------+------------+--------------+
|Eltingville/Annad...|          84|             1|
|       Arden Heights|           5|             1|
|Governor's Island...|         105|             1|
|       Port Richmond|         187|             3|
| Green-Wood Cemetery|         111|             4|
|         Great Kills|         109|             4|
|   Rossville/Woodrow|         204|             4|
|       Rikers Island|         199|             4|
|         Jamaica Bay|           2|             5|
|         Westerleigh|         251|            12|
|        Crotona Park|          59|            14|
|             Oakwood|         176|            14|
|       West Brighton|         245|            14|
|New Dorp/Midland ...|         172|            14|
|       Willets Point|         253|            15|
|Breezy Point/Fort...|          27|            16|
|Saint George/New ...|         